# Dos features para mi proyecto (M5, CA_1)

Proyecto: demanda de productos de Walmart en la tienda CA_1. Archivo: `../data/processed/obt_sales_CA_1.csv.gz` (el mismo de la exploración).

## Paso 1: el momento de la predicción

El modelo predice las unidades vendidas de un producto **mañana**. Se predice al cierre de hoy: existen las ventas de cada producto hasta hoy, el precio vigente y el calendario de mañana (día de la semana, eventos, SNAP). Todavía no existen las ventas de mañana ni su `revenue` (se calcula con `units`).

In [1]:
import pandas as pd

datos = pd.read_csv("../data/processed/obt_sales_CA_1.csv.gz",
                    usecols=["item_id", "d_num", "units", "is_pre_launch"])
datos = datos.query("not is_pre_launch").sort_values(["item_id", "d_num"]).copy()
OBJETIVO = "units"

## Paso 2: feature 1, ventas de hace 7 días

In [2]:
# Unidades del mismo producto hace 7 dias (mismo dia de la semana, semana pasada).
datos["units_lag_7"] = datos.groupby("item_id")[OBJETIVO].shift(7)

# Evidencia: correlacion con el objetivo.
datos[["units_lag_7", OBJETIVO]].corr()

,units_lag_7,units
units_lag_7,1.000000,0.710061
units,0.710061,1.000000


**Feature 1: `units_lag_7`.** Sale del hallazgo de que el departamento y el nivel de ventas definen la serie, y del conocimiento del negocio: la demanda tiene ciclo semanal (lo que se vendió hace una semana, el mismo día, anticipa lo de mañana). **Evidencia:** correlación 0.71 con `units`. **Existe al predecir:** sí, son ventas de hace 7 días, ya cerradas. **Codificación:** numérica; se escala en el pipeline.

## Paso 3: feature 2, promedio de los 28 días previos

In [3]:
# Promedio de unidades del producto en los 28 dias anteriores (sin incluir el dia a predecir).
datos["units_media_28"] = datos.groupby("item_id")[OBJETIVO].transform(lambda s: s.shift(1).rolling(28).mean())

# Evidencia: correlacion con el objetivo, y comparacion con la feature 1.
datos[["units_lag_7", "units_media_28", OBJETIVO]].corr()[OBJETIVO]

units_lag_7       0.710061
units_media_28    0.790248
units             1.000000
Name: units, dtype: float64

**Feature 2: `units_media_28`.** Sale del hallazgo de que el objetivo es muy asimétrico, con muchos ceros (mediana 0): una sola semana es ruidosa, y el promedio de 4 semanas suaviza ese ruido y captura el nivel del producto. **Evidencia:** correlación 0.79 con `units`, mayor que la de `units_lag_7` (0.71). **Existe al predecir:** sí, usa ventas hasta ayer (`shift(1)`), nunca la del día a predecir. **Codificación:** numérica; se escala en el pipeline. Las dos features pasan a `pipeline_proyecto.py`.

In [4]:
datos.dropna().head()

,d_num,item_id,units,is_pre_launch,units_lag_7,units_media_28
28,29,FOODS_1_001,2,False,1.0,1.392857
29,30,FOODS_1_001,2,False,2.0,1.357143
30,31,FOODS_1_001,0,False,0.0,1.428571
31,32,FOODS_1_001,2,False,2.0,1.428571
32,33,FOODS_1_001,1,False,2.0,1.464286
